# populariteit bepalen van tedtalk videos

# eerst de data ophalen en opschonen

In [65]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
import joblib

df = pd.read_csv("Kaggle_TED_video_metadata_balanced.csv")
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          600 non-null    object
 1   tags           590 non-null    object
 2   views          600 non-null    int64 
 3   likes          600 non-null    int64 
 4   dislikes       600 non-null    int64 
 5   comment_count  600 non-null    int64 
 6   published_at   600 non-null    object
 7   duration       600 non-null    object
 8   category_id    600 non-null    int64 
dtypes: int64(5), object(4)
memory usage: 42.3+ KB


,views,likes,dislikes,comment_count,category_id
count,6.000000e+02,6.000000e+02,600.0,600.000000,600.000000
mean,1.066151e+06,2.263473e+04,0.0,1070.870000,25.030000
std,3.611008e+06,1.000825e+05,0.0,4055.503954,4.812244
min,1.000000e+00,0.000000e+00,0.0,0.000000,1.000000
25%,5.922150e+04,6.242500e+02,0.0,86.000000,22.000000
50%,1.332595e+05,1.846000e+03,0.0,202.000000,27.000000
75%,4.101668e+05,6.382250e+03,0.0,511.750000,28.000000
max,5.562224e+07,1.921445e+06,0.0,77980.000000,29.000000


##### voor de zekerheid verwijder ik gelijk alle duplicaten.

##### de title is nutteloos voor machine learning, dus die halen we weg. Ook zijn de dislikes allemaal 0, omdat youtube dit uitgeschakeld heeft. Daarom verwijder ik deze column ook.

In [66]:
df = df.drop(columns=["title", "dislikes"], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   tags           590 non-null    object
 1   views          600 non-null    int64 
 2   likes          600 non-null    int64 
 3   comment_count  600 non-null    int64 
 4   published_at   600 non-null    object
 5   duration       600 non-null    object
 6   category_id    600 non-null    int64 
dtypes: int64(4), object(3)
memory usage: 32.9+ KB


##### de column "tags" bevat 10 null waarden. Hier ga ik echter niks aan doen, omdat dat natuurlijk voorkomt in de dataset. Er worden in de realiteit video's geopload zonder tags. Verder zijn er geen null waarden.

In [67]:
df.head(5)

,tags,views,likes,comment_count,published_at,duration,category_id
0,"TED Talk,TED Talks,Children,Community,Death,Fa...",77455,1768,49,2017-03-24T15:32:48Z,PT15M19S,29
1,"TEDTalk,TEDTalks,Addiction,Computers,Interface...",800326,20579,572,2017-08-01T15:29:04Z,PT9M30S,22
2,"TEDTalk,TEDTalks,Cancer,Community,Compassion,D...",87635,1877,48,2017-05-30T18:17:56Z,PT10M49S,22
3,"TEDTalk,TEDTalks,Children,Communication,Commun...",190840,4726,187,2017-04-12T15:17:51Z,PT11M56S,22
4,"TEDTalk,TEDTalks,Children,Global issues,Humani...",98523,2669,226,2017-07-25T15:06:25Z,PT14M14S,29


##### de views, likes en comment_count moeten gescaled worden voor optimalisatie

In [68]:
scaler = StandardScaler()
df[["views", "likes", "comment_count"]] = scaler.fit_transform(df[["views", "likes", "comment_count"]])
df.head(5)

,tags,views,likes,comment_count,published_at,duration,category_id
0,"TED Talk,TED Talks,Children,Community,Death,Fa...",-0.274029,-0.208669,-0.252181,2017-03-24T15:32:48Z,PT15M19S,29
1,"TEDTalk,TEDTalks,Addiction,Computers,Interface...",-0.073677,-0.020557,-0.123113,2017-08-01T15:29:04Z,PT9M30S,22
2,"TEDTalk,TEDTalks,Cancer,Community,Compassion,D...",-0.271208,-0.207579,-0.252428,2017-05-30T18:17:56Z,PT10M49S,22
3,"TEDTalk,TEDTalks,Children,Communication,Commun...",-0.242603,-0.179089,-0.218125,2017-04-12T15:17:51Z,PT11M56S,22
4,"TEDTalk,TEDTalks,Children,Global issues,Humani...",-0.268190,-0.199659,-0.208501,2017-07-25T15:06:25Z,PT14M14S,29


##### De tag column is nog niet leesbaar voor machine learning. Dit is nominale data dus ik maak gebruik van one-hot encoding

In [69]:
df = pd.get_dummies(df, columns=["tags"], drop_first=True)
df.head(5)

,views,likes,comment_count,published_at,duration,category_id,"tags_Academic,Adjective,Admission,Adverb,Art,Business,Cambridge English,Collocation,Economics,English,Englist,Englist.me,Entertainment,Entrepreneur,Environment,Expression,FLuent,GMAT,GRE,Global Issues,IELTS,Learner,Learning,Linguistics,Meaning,Medical,News,Noun,Pedagogy,Phrase,Practical,SAT,STEAM,STEM,School,Science,Sentence,Society,Study,TED,TED Talks,TED-Ed,TOEFL,TOEIC,TV,Teaching,Technology,The Economist,United States,University,Verb,Vocabulary,Vocabulary Building,Word List","tags_Advertising,Marketing Communications,Creative,Video,Strategy,Content Creation,Production,Generate,Insights,Media,Technical,Digital,Skills,Network","tags_Al,Gore,TED,TEDTalks,Talks,climate,crisis,environment,global,warming,An Inconvenient Truth","tags_Alan,Kay,ted,tedtalks,education,technology,children,collaboration,computers,design",...,"tags_ted talk,usa,TEDxorangecoast,english,ted,tedx,tedx talk,ted x,living beyond limits,ted talks,tedx talks,adaptive action sports,orange coast,amy purdy","tags_ted x,ted,tedx talk,tedx,ted talks,tedx talks,TEDx,ted talk,Andrew Solomon (Author),depression,mental health,tedxmet,andrew solomon","tags_ted x,tedx talks,tedx,ted talks,TEDxMidAtlantic,ted,ted talk,TEDx,tedx talk","tags_ted,tedtalks,Mallika Sarabhai,TEDIndia,India,talks,art,music,dance,change,politics,storytelling","tags_ted,tedtalks,talks,ground,zero,david,rockwell,architecture,9/11","tags_tedx,ted x,tedx talks,tedx talk,food,ted,ted talk,TEDx,Asheville,NextGeneration,Baehr,Birke,organic,ted talks","tags_the universal translator,universal translator,translator,does that make sense,language,discovery,travel tech","tags_women in the world,tina brown live media,tina brown,women in the world summit,new york city,Lincoln center,womens empowerment,feminism,women of color,maternal mortality,ancient issue,local solutions,Liya Kebede,Supermodel,Ivy Prosper,Zubaida Bai,ayzh,Mary Goretti Musoke,Alyse Nelson,birth,women's health,maternal health","tags_zeitgeist,ted talks,conferences,tech,business,arts,google,museums,digital museum,museum of the future,the people's museum,Amit Sood,Bringing Museums to the Internet","tags_курсы иностранных языков,английский,немецкий,французский,испанский,итальянский,японский,китайский,курсы харьков,Dean Kamen (Organization Leader),Invention (Literature Subject),английский харьков"
0,-0.274029,-0.208669,-0.252181,2017-03-24T15:32:48Z,PT15M19S,29,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,-0.073677,-0.020557,-0.123113,2017-08-01T15:29:04Z,PT9M30S,22,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,-0.271208,-0.207579,-0.252428,2017-05-30T18:17:56Z,PT10M49S,22,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,-0.242603,-0.179089,-0.218125,2017-04-12T15:17:51Z,PT11M56S,22,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,-0.268190,-0.199659,-0.208501,2017-07-25T15:06:25Z,PT14M14S,29,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
